
# Buy Fee Model Dataset Builder v6.1

## Purpose

V6.1 corrects and validates the V6 data pipeline before any model training.

### Locked business rules

**Sold**
- `VRSTATUS == "HST"` AND `VRINVOICE != "RETURN"`
- OR `VRSTATUS == "SLD"` AND `VRIFORTRY != "Y"`

**Important V6.1 fix:** blank / NULL `VRINVOICE` and `VRIFORTRY` values are explicitly converted to empty strings before applying the logic. Therefore:
- HST + blank invoice = sold
- SLD + blank VRIFORTRY = sold

**Inventory**
- HST / SLD / NOS are retained
- INV and SAL are excluded

**Price tier**
- `SALEAMT_FINAL` only, for sold and unsold units

**Buy Fee / RPU**
- `BUYFEE_FINAL` only
- Buyer_BuyFee is never used
- Buy Fee revenue is counted only on sold units

**Primary event eligibility**
- 8-week pre/post window
- >= 6 observed weeks on each side
- >= 25 registered units on each side

### New diagnostics in v6.1
- Reconcile expected ~52 locations vs source locations
- Location-level VIN/status/date/event counts
- NULL-safe sold reconciliation by VRSTATUS
- BUYFEE_FINAL NULL / zero / positive coverage by status and sold flag
- Exact counts before and after each model-ready filter
- Re-run the full V6 4W / 8W / 12W event pipeline


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings

warnings.filterwarnings("ignore")

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

INPUT_CSV = DATA_FOLDER / "VIN_LEVEL_DATA_WITH_DATES_AND_SALE_PRICE_NEW_DATA.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_dataset_v6_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 250_000
ENCODING = "cp1252"

PRICE_TIER_WIDTH = 1000
WINDOWS = [4, 8, 12]
PRIMARY_WINDOW = 8

MIN_REGISTERED_PRE = 25
MIN_REGISTERED_POST = 25
MIN_OBSERVED_WEEKS_PRE = 6
MIN_OBSERVED_WEEKS_POST = 6

MIN_SOLD_PRE_RPU = 15
MIN_SOLD_POST_RPU = 15
MIN_BUYFEE_COVERAGE = 0.90

OVERLAP_LOOKBACK_WEEKS = 8
OVERLAP_LOOKAHEAD_WEEKS = 8

EXPECTED_LOCATION_COUNT = 52

print("=" * 80)
print("V6.1 CONFIGURATION")
print("=" * 80)
print("Input:", INPUT_CSV)
print("Exists:", INPUT_CSV.exists())
print("Output:", OUTPUT_DIR)

if not INPUT_CSV.exists():
    print("\nCSV files found:")
    for f in DATA_FOLDER.glob("*.csv"):
        print(" -", f.name)
    raise FileNotFoundError(INPUT_CSV)


V6.1 CONFIGURATION
Input: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\VIN_LEVEL_DATA_WITH_DATES_AND_SALE_PRICE_NEW_DATA.csv
Exists: True
Output: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_dataset_v6_1


## 1. Column mapping

In [2]:

header_df = pd.read_csv(
    INPUT_CSV,
    nrows=10,
    encoding=ENCODING,
    encoding_errors="replace",
    low_memory=False
)

ALIASES = {
    "vin": ["VRVIN", "VIN"],
    "status": ["VRSTATUS"],
    "sale_date": ["VRSALEDATE", "SALEDATE"],
    "saleamt_final": ["SALEAMT_FINAL"],
    "location": ["VRLOCATION", "LOCATION", "Location"],
    "lane": ["LANE", "Lane", "ACTUAL_LANE"],
    "invoice": ["VRINVOICE"],
    "ifortry": ["VRIFORTRY"],
    "buyfee_final": ["BUYFEE_FINAL", "BUYFEE-FINAL", "BUYFEE FINAL"],
    "change_status": ["BUYFEE_CHANGE_STATUS"],
    "change_date": ["BUYFEE_CHANGE_DATE"],
    "previous_buyfee": ["PREVIOUS_BUYFEE"],
    "change_amount": ["BUY_FEE_CHANGE_AMOUNT"],
    "vehicle_year": ["Vehicle_year", "VEHICLE_YEAR"],
    "mileage": ["VRMILEAGE", "MILEAGE"],
    "make": ["Make", "MAKE"],
    "condition1": ["Vehicle_condition_grade_1", "CONDITION_GRADE_1"],
    "condition2": ["Vehicle_condition_grade_2", "CONDITION_GRADE_2"],
    "condition_overall": ["Vehicle_condition_overall", "CONDITION_OVERALL"],
    "high_bid": ["VRHIGHBID", "HIGH_BID"],
    "floor": ["VRFLOOR", "FLOOR"],
}

def resolve_col(columns, aliases):
    lookup = {str(c).strip().upper(): c for c in columns}
    for alias in aliases:
        k = alias.strip().upper()
        if k in lookup:
            return lookup[k]
    return None

COL = {k: resolve_col(header_df.columns, v) for k, v in ALIASES.items()}

display(pd.DataFrame(
    [{"Logical_Field": k, "Resolved_Column": v} for k, v in COL.items()]
))

required = [
    "status", "sale_date", "saleamt_final", "location", "lane",
    "invoice", "ifortry", "buyfee_final",
    "change_status", "change_date", "previous_buyfee", "change_amount"
]

missing = [x for x in required if COL[x] is None]

if missing:
    raise ValueError("Missing required columns: " + ", ".join(missing))


,Logical_Field,Resolved_Column
0,vin,VRVIN
1,status,VRSTATUS
2,sale_date,VRSALEDATE
3,saleamt_final,SALEAMT_FINAL
4,location,VRLOCATION
5,lane,LANE
6,invoice,VRINVOICE
7,ifortry,VRIFORTRY
8,buyfee_final,BUYFEE-FINAL
9,change_status,BUYFEE_CHANGE_STATUS


## 2. Helper functions — NULL-safe sold logic

In [3]:

def clean_text(s):
    return s.astype("string").str.strip()

def clean_upper_no_null(s):
    # Critical V6.1 rule: blank / NULL is a valid non-RETURN / non-Y condition.
    return (
        s.astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def parse_date_series(s):
    x = s.astype("string").str.strip()
    x2 = x.str.replace(r"\.0$", "", regex=True)

    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    m8 = x2.str.fullmatch(r"\d{8}", na=False)

    if m8.any():
        out.loc[m8] = pd.to_datetime(
            x2.loc[m8], format="%Y%m%d", errors="coerce"
        )

    if (~m8).any():
        out.loc[~m8] = pd.to_datetime(
            x.loc[~m8], errors="coerce"
        )

    return out

def exact_sold_flag(df):
    status = clean_upper_no_null(df[COL["status"]])
    invoice = clean_upper_no_null(df[COL["invoice"]])
    ifortry = clean_upper_no_null(df[COL["ifortry"]])

    return (
        (status.eq("HST") & invoice.ne("RETURN"))
        |
        (status.eq("SLD") & ifortry.ne("Y"))
    ).astype(bool)

def valid_inventory_flag(df):
    status = clean_upper_no_null(df[COL["status"]])
    return status.isin(["HST", "SLD", "NOS"])

def make_price_tier(saleamt_final):
    amt = pd.to_numeric(saleamt_final, errors="coerce")
    low = np.floor(amt / PRICE_TIER_WIDTH) * PRICE_TIER_WIDTH
    high = low + PRICE_TIER_WIDTH

    label = pd.Series(pd.NA, index=amt.index, dtype="string")
    valid = amt.notna() & (amt >= 0)

    label.loc[valid] = (
        low.loc[valid].astype("Int64").astype(str)
        + "-"
        + high.loc[valid].astype("Int64").astype(str)
    )

    return low, high, label

def safe_pct_change(post, pre):
    if pd.isna(pre) or pre == 0 or pd.isna(post):
        return np.nan
    return (post - pre) / pre

def linear_slope(values):
    vals = np.asarray(values, dtype=float)
    mask = np.isfinite(vals)

    if mask.sum() < 2:
        return np.nan

    return np.polyfit(
        np.arange(len(vals))[mask],
        vals[mask],
        1
    )[0]

def coefficient_of_variation(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) < 2:
        return np.nan

    mean = vals.mean()

    if mean == 0:
        return np.nan

    return vals.std(ddof=1) / abs(mean)


## 3. Chunked source audit + V6 pipeline inputs

In [4]:

weekly_parts = []
event_parts = []
location_parts = []
sold_audit_parts = []
fee_audit_parts = []

rows_processed = 0

reader = pd.read_csv(
    INPUT_CSV,
    chunksize=CHUNK_SIZE,
    encoding=ENCODING,
    encoding_errors="replace",
    low_memory=False
)

for chunk_num, chunk in enumerate(reader, start=1):

    rows_processed += len(chunk)

    status = clean_upper_no_null(chunk[COL["status"]])
    invoice = clean_upper_no_null(chunk[COL["invoice"]])
    ifortry = clean_upper_no_null(chunk[COL["ifortry"]])

    sale_dt = parse_date_series(chunk[COL["sale_date"]])
    saleamt_final = to_num(chunk[COL["saleamt_final"]])

    loc = clean_upper_no_null(chunk[COL["location"]])
    lane = clean_upper_no_null(chunk[COL["lane"]])

    buyfee = to_num(chunk[COL["buyfee_final"]])

    change_status = clean_upper_no_null(chunk[COL["change_status"]])
    change_date = parse_date_series(chunk[COL["change_date"]])
    prev_fee = to_num(chunk[COL["previous_buyfee"]])
    change_amt = to_num(chunk[COL["change_amount"]])

    sold = exact_sold_flag(chunk)
    valid_inventory = valid_inventory_flag(chunk)

    tier_low, tier_high, tier_label = make_price_tier(saleamt_final)

    week_start = (
        sale_dt
        - pd.to_timedelta(sale_dt.dt.weekday, unit="D")
    )

    # --------------------------------------------------------
    # Sold logic audit
    # --------------------------------------------------------

    sold_audit = (
        pd.DataFrame({
            "VRSTATUS": status,
            "Invoice_Blank": invoice.eq(""),
            "Invoice_RETURN": invoice.eq("RETURN"),
            "IFORTRY_Blank": ifortry.eq(""),
            "IFORTRY_Y": ifortry.eq("Y"),
            "Sold_Flag": sold,
        })
        .groupby("VRSTATUS", dropna=False)
        .agg(
            Raw_Rows=("VRSTATUS", "size"),
            Invoice_Blank=("Invoice_Blank", "sum"),
            Invoice_RETURN=("Invoice_RETURN", "sum"),
            IFORTRY_Blank=("IFORTRY_Blank", "sum"),
            IFORTRY_Y=("IFORTRY_Y", "sum"),
            Sold_Flag_Count=("Sold_Flag", "sum"),
        )
        .reset_index()
    )

    sold_audit_parts.append(sold_audit)

    # --------------------------------------------------------
    # BUYFEE_FINAL audit
    # --------------------------------------------------------

    fee_state = np.select(
        [
            buyfee.isna(),
            buyfee.eq(0),
            buyfee.gt(0),
            buyfee.lt(0),
        ],
        [
            "NULL",
            "ZERO",
            "POSITIVE",
            "NEGATIVE",
        ],
        default="OTHER"
    )

    fee_audit = (
        pd.DataFrame({
            "VRSTATUS": status,
            "Sold_Flag": sold,
            "BuyFee_State": fee_state,
        })
        .groupby(
            ["VRSTATUS", "Sold_Flag", "BuyFee_State"],
            dropna=False
        )
        .size()
        .reset_index(name="Count")
    )

    fee_audit_parts.append(fee_audit)

    # --------------------------------------------------------
    # Location reconciliation audit
    # --------------------------------------------------------

    loc_df = pd.DataFrame({
        "VRLOCATION": loc,
        "VRSTATUS": status,
        "Sale_Date": sale_dt,
        "Sold_Flag": sold,
        "Valid_Inventory": valid_inventory,
        "Is_Change": change_status.eq("CHANGE"),
    })

    loc_df = loc_df[loc_df["VRLOCATION"].ne("")]

    loc_agg = (
        loc_df.groupby("VRLOCATION", dropna=False)
        .agg(
            Total_Rows=("VRLOCATION", "size"),
            Valid_Inventory_Rows=("Valid_Inventory", "sum"),
            Sold_Rows=("Sold_Flag", "sum"),
            HST_Rows=("VRSTATUS", lambda s: (s == "HST").sum()),
            SLD_Rows=("VRSTATUS", lambda s: (s == "SLD").sum()),
            NOS_Rows=("VRSTATUS", lambda s: (s == "NOS").sum()),
            INV_Rows=("VRSTATUS", lambda s: (s == "INV").sum()),
            SAL_Rows=("VRSTATUS", lambda s: (s == "SAL").sum()),
            Change_Source_Rows=("Is_Change", "sum"),
            First_Date=("Sale_Date", "min"),
            Last_Date=("Sale_Date", "max"),
        )
        .reset_index()
    )

    location_parts.append(loc_agg)

    # --------------------------------------------------------
    # Optional mix fields
    # --------------------------------------------------------

    vehicle_year = (
        to_num(chunk[COL["vehicle_year"]])
        if COL["vehicle_year"]
        else pd.Series(np.nan, index=chunk.index)
    )

    mileage = (
        to_num(chunk[COL["mileage"]])
        if COL["mileage"]
        else pd.Series(np.nan, index=chunk.index)
    )

    high_bid = (
        to_num(chunk[COL["high_bid"]])
        if COL["high_bid"]
        else pd.Series(np.nan, index=chunk.index)
    )

    floor = (
        to_num(chunk[COL["floor"]])
        if COL["floor"]
        else pd.Series(np.nan, index=chunk.index)
    )

    cond_candidates = []

    for k in ["condition_overall", "condition1", "condition2"]:
        if COL[k]:
            cond_candidates.append(to_num(chunk[COL[k]]))

    if cond_candidates:
        condition = (
            pd.concat(cond_candidates, axis=1)
            .bfill(axis=1)
            .iloc[:, 0]
        )
    else:
        condition = pd.Series(np.nan, index=chunk.index)

    make = (
        clean_upper_no_null(chunk[COL["make"]])
        if COL["make"]
        else pd.Series("", index=chunk.index)
    )

    vehicle_age = sale_dt.dt.year - vehicle_year

    # --------------------------------------------------------
    # Weekly panel source
    # --------------------------------------------------------

    keep = (
        valid_inventory
        & sale_dt.notna()
        & saleamt_final.notna()
        & saleamt_final.ge(0)
        & loc.ne("")
        & lane.ne("")
        & tier_label.notna()
    )

    temp = pd.DataFrame({
        "Week_Start": week_start,
        "VRLOCATION": loc,
        "LANE": lane,
        "price_tier": tier_label,
        "price_tier_low": tier_low,
        "price_tier_high": tier_high,

        "Registered_Units": 1,
        "Sold_Units": sold.astype("int8"),
        "NOS_Units": status.eq("NOS").astype("int8"),

        "SaleAmt_Final": saleamt_final,

        "BuyFee_Final_Sold": buyfee.where(sold, np.nan),
        "Sold_With_BuyFee": (sold & buyfee.notna()).astype("int8"),
        "Sold_With_Positive_BuyFee": (sold & buyfee.gt(0)).astype("int8"),
        "Sold_With_Zero_BuyFee": (sold & buyfee.eq(0)).astype("int8"),

        "Vehicle_Age": vehicle_age,
        "Mileage": mileage,
        "Condition": condition,
        "High_Bid": high_bid,
        "Floor": floor,
        "Make": make,
    })

    temp = temp.loc[keep].copy()

    weekly_chunk = (
        temp.groupby(
            [
                "Week_Start",
                "VRLOCATION",
                "LANE",
                "price_tier",
                "price_tier_low",
                "price_tier_high"
            ],
            dropna=False,
            sort=False
        )
        .agg(
            Registered_Units=("Registered_Units", "sum"),
            Sold_Units=("Sold_Units", "sum"),
            NOS_Units=("NOS_Units", "sum"),

            SaleAmt_Final_Sum=("SaleAmt_Final", "sum"),
            SaleAmt_Final_Count=("SaleAmt_Final", "count"),

            BuyFee_Revenue=("BuyFee_Final_Sold", "sum"),
            BuyFee_Final_Count=("BuyFee_Final_Sold", "count"),
            Sold_With_BuyFee=("Sold_With_BuyFee", "sum"),
            Sold_With_Positive_BuyFee=("Sold_With_Positive_BuyFee", "sum"),
            Sold_With_Zero_BuyFee=("Sold_With_Zero_BuyFee", "sum"),

            Vehicle_Age_Sum=("Vehicle_Age", "sum"),
            Vehicle_Age_Count=("Vehicle_Age", "count"),

            Mileage_Sum=("Mileage", "sum"),
            Mileage_Count=("Mileage", "count"),

            Condition_Sum=("Condition", "sum"),
            Condition_Count=("Condition", "count"),

            High_Bid_Sum=("High_Bid", "sum"),
            High_Bid_Count=("High_Bid", "count"),

            Floor_Sum=("Floor", "sum"),
            Floor_Count=("Floor", "count"),

            Unique_Makes=("Make", "nunique"),
        )
        .reset_index()
    )

    weekly_parts.append(weekly_chunk)

    # --------------------------------------------------------
    # Actual fee-change events
    # --------------------------------------------------------

    event_mask = (
        change_status.eq("CHANGE")
        & valid_inventory
        & saleamt_final.notna()
        & loc.ne("")
        & lane.ne("")
        & change_date.notna()
        & tier_label.notna()
    )

    if event_mask.any():

        ev = pd.DataFrame({
            "VRLOCATION": loc.loc[event_mask],
            "LANE": lane.loc[event_mask],
            "price_tier": tier_label.loc[event_mask],
            "price_tier_low": tier_low.loc[event_mask],
            "price_tier_high": tier_high.loc[event_mask],
            "BUYFEE_CHANGE_DATE": change_date.loc[event_mask],
            "PREVIOUS_BUYFEE": prev_fee.loc[event_mask],
            "BUYFEE_CHANGE_AMOUNT": change_amt.loc[event_mask],
        })

        ev["NEW_BUYFEE"] = (
            ev["PREVIOUS_BUYFEE"]
            + ev["BUYFEE_CHANGE_AMOUNT"]
        )

        event_parts.append(ev)

    if chunk_num % 5 == 0:
        print(
            f"Processed {chunk_num:,} chunks | "
            f"{rows_processed:,} source rows"
        )

    del chunk, temp, weekly_chunk, loc_df
    gc.collect()

print("\nSOURCE PASS COMPLETE")
print("Rows processed:", f"{rows_processed:,}")


Processed 5 chunks | 1,250,000 source rows
Processed 10 chunks | 2,500,000 source rows
Processed 15 chunks | 3,750,000 source rows
Processed 20 chunks | 5,000,000 source rows

SOURCE PASS COMPLETE
Rows processed: 5,540,598


## 4. Sold logic reconciliation

In [5]:

sold_recon = (
    pd.concat(sold_audit_parts, ignore_index=True)
    .groupby("VRSTATUS", as_index=False)
    .sum(numeric_only=True)
    .sort_values("Raw_Rows", ascending=False)
)

sold_recon["Expected_Sold_From_Status_Rules"] = np.select(
    [
        sold_recon["VRSTATUS"].eq("HST"),
        sold_recon["VRSTATUS"].eq("SLD"),
    ],
    [
        sold_recon["Raw_Rows"] - sold_recon["Invoice_RETURN"],
        sold_recon["Raw_Rows"] - sold_recon["IFORTRY_Y"],
    ],
    default=0
)

sold_recon["Sold_Reconciliation_Difference"] = (
    sold_recon["Sold_Flag_Count"]
    - sold_recon["Expected_Sold_From_Status_Rules"]
)

sold_recon.to_csv(
    OUTPUT_DIR / "01_SOLD_LOGIC_RECONCILIATION_V6_1.csv",
    index=False
)

display(sold_recon)

if (sold_recon["Sold_Reconciliation_Difference"] != 0).any():
    print("WARNING: sold flag does not fully reconcile to the stated business rule.")
else:
    print("PASS: sold flag fully reconciles to the stated HST/SLD business rule.")


,VRSTATUS,Raw_Rows,Invoice_Blank,Invoice_RETURN,IFORTRY_Blank,IFORTRY_Y,Sold_Flag_Count,Expected_Sold_From_Status_Rules,Sold_Reconciliation_Difference
0,HST,3263450,3217123,45832,1,0,3217618,3217618,0
2,NOS,2265049,2168843,95400,2219174,1,0,0,0
3,SAL,8548,8548,0,8548,0,0,0,0
1,INV,3389,3389,0,3388,0,0,0,0
4,SLD,162,162,0,0,8,154,154,0


PASS: sold flag fully reconciles to the stated HST/SLD business rule.


## 5. BUYFEE_FINAL coverage validation

In [6]:

fee_audit = (
    pd.concat(fee_audit_parts, ignore_index=True)
    .groupby(
        ["VRSTATUS", "Sold_Flag", "BuyFee_State"],
        as_index=False
    )["Count"]
    .sum()
)

fee_pivot = (
    fee_audit.pivot_table(
        index=["VRSTATUS", "Sold_Flag"],
        columns="BuyFee_State",
        values="Count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for c in ["NULL", "ZERO", "POSITIVE", "NEGATIVE", "OTHER"]:
    if c not in fee_pivot.columns:
        fee_pivot[c] = 0

fee_pivot["All_Count"] = (
    fee_pivot[["NULL", "ZERO", "POSITIVE", "NEGATIVE", "OTHER"]]
    .sum(axis=1)
)

fee_pivot["Populated_Count"] = (
    fee_pivot["All_Count"]
    - fee_pivot["NULL"]
)

fee_pivot["Pct_Populated"] = np.where(
    fee_pivot["All_Count"] > 0,
    fee_pivot["Populated_Count"] / fee_pivot["All_Count"],
    np.nan
)

fee_pivot["Pct_Positive"] = np.where(
    fee_pivot["All_Count"] > 0,
    fee_pivot["POSITIVE"] / fee_pivot["All_Count"],
    np.nan
)

fee_pivot.to_csv(
    OUTPUT_DIR / "02_BUYFEE_FINAL_COVERAGE_V6_1.csv",
    index=False
)

display(fee_pivot)

sold_fee = fee_pivot[fee_pivot["Sold_Flag"].eq(True)]

print("\nSold-unit BUYFEE_FINAL totals:")
print("Sold rows:", f"{sold_fee['All_Count'].sum():,}")
print("Populated:", f"{sold_fee['Populated_Count'].sum():,}")
print("Positive :", f"{sold_fee['POSITIVE'].sum():,}")
print("Zero     :", f"{sold_fee['ZERO'].sum():,}")
print("Null     :", f"{sold_fee['NULL'].sum():,}")


BuyFee_State,VRSTATUS,Sold_Flag,NULL,POSITIVE,ZERO,NEGATIVE,OTHER,All_Count,Populated_Count,Pct_Populated,Pct_Positive
0,HST,False,41260,4561,11,0,0,45832,4572,0.099756,0.099516
1,HST,True,1193411,2016096,8111,0,0,3217618,2024207,0.629101,0.626580
2,INV,False,3360,29,0,0,0,3389,29,0.008557,0.008557
3,NOS,False,2054933,209772,344,0,0,2265049,210116,0.092764,0.092613
4,SAL,False,8412,130,6,0,0,8548,136,0.015910,0.015208
5,SLD,False,7,1,0,0,0,8,1,0.125000,0.125000
6,SLD,True,113,41,0,0,0,154,41,0.266234,0.266234



Sold-unit BUYFEE_FINAL totals:
Sold rows: 3,217,772
Populated: 2,024,248
Positive : 2,016,137
Zero     : 8,111
Null     : 1,193,524


## 6. Location reconciliation: 54 vs expected 52

In [7]:

location_audit = (
    pd.concat(location_parts, ignore_index=True)
    .groupby("VRLOCATION", as_index=False)
    .agg(
        Total_Rows=("Total_Rows", "sum"),
        Valid_Inventory_Rows=("Valid_Inventory_Rows", "sum"),
        Sold_Rows=("Sold_Rows", "sum"),
        HST_Rows=("HST_Rows", "sum"),
        SLD_Rows=("SLD_Rows", "sum"),
        NOS_Rows=("NOS_Rows", "sum"),
        INV_Rows=("INV_Rows", "sum"),
        SAL_Rows=("SAL_Rows", "sum"),
        Change_Source_Rows=("Change_Source_Rows", "sum"),
        First_Date=("First_Date", "min"),
        Last_Date=("Last_Date", "max"),
    )
    .sort_values(["Valid_Inventory_Rows", "Total_Rows"], ascending=False)
    .reset_index(drop=True)
)

location_audit["Has_Valid_Inventory"] = (
    location_audit["Valid_Inventory_Rows"] > 0
)

location_audit["Has_Fee_Change_Source_Rows"] = (
    location_audit["Change_Source_Rows"] > 0
)

location_audit.to_csv(
    OUTPUT_DIR / "03_LOCATION_RECONCILIATION_V6_1.csv",
    index=False
)

valid_locations = location_audit[
    location_audit["Has_Valid_Inventory"]
].copy()

print("All location labels:", location_audit["VRLOCATION"].nunique())
print("Locations with HST/SLD/NOS inventory:", valid_locations["VRLOCATION"].nunique())
print("Expected business location count:", EXPECTED_LOCATION_COUNT)
print(
    "Difference:",
    valid_locations["VRLOCATION"].nunique() - EXPECTED_LOCATION_COUNT
)

print("\nFull valid-location list:")
display(
    valid_locations[
        [
            "VRLOCATION",
            "Valid_Inventory_Rows",
            "Sold_Rows",
            "HST_Rows",
            "SLD_Rows",
            "NOS_Rows",
            "Change_Source_Rows",
            "First_Date",
            "Last_Date",
        ]
    ]
)

print(
    "\nReview the smallest/legacy-looking locations first when reconciling "
    "the two extra source labels."
)


All location labels: 54
Locations with HST/SLD/NOS inventory: 54
Expected business location count: 52
Difference: 2

Full valid-location list:


,VRLOCATION,Valid_Inventory_Rows,Sold_Rows,HST_Rows,SLD_Rows,NOS_Rows,Change_Source_Rows,First_Date,Last_Date
0,BIRMINGHAM,286435,154656,160740,0,125695,47,2022-12-26,2026-09-30
1,TULSA,229643,137378,138919,1,90723,1596,2022-12-27,2026-10-02
2,COLUMBUS,222529,132153,135070,0,87459,1045,2022-12-28,2026-10-14
3,SAINTLOUIS,214895,117495,119681,0,95214,115,2022-12-27,2026-10-01
4,KANSAS,209141,125311,128066,24,81051,988,2022-12-27,2026-10-01
5,HARRISBURG,200925,120555,121872,0,79053,1128,2022-12-27,2026-10-01
6,DALLAS,199482,120858,122026,22,77434,0,2022-12-26,2026-10-01
7,ATLANTA,194795,96632,98701,0,96094,1164,2022-12-27,2026-12-25
8,DENVER,190547,103063,103521,0,87026,4379,2022-12-29,2026-10-01
9,WESTMICHIGAN,185011,106679,107434,0,77577,912,2022-12-27,2026-10-02



Review the smallest/legacy-looking locations first when reconciling the two extra source labels.


## 7. Rebuild weekly panel with corrected sold logic

In [8]:

weekly_raw = pd.concat(weekly_parts, ignore_index=True)

group_cols = [
    "Week_Start",
    "VRLOCATION",
    "LANE",
    "price_tier",
    "price_tier_low",
    "price_tier_high"
]

sum_cols = [
    c for c in weekly_raw.columns
    if c not in group_cols
]

weekly = (
    weekly_raw.groupby(
        group_cols,
        dropna=False,
        sort=False
    )[sum_cols]
    .sum(min_count=1)
    .reset_index()
)

weekly["Conversion"] = np.where(
    weekly["Registered_Units"] > 0,
    weekly["Sold_Units"] / weekly["Registered_Units"],
    np.nan
)

weekly["Avg_SaleAmt_Final"] = np.where(
    weekly["SaleAmt_Final_Count"] > 0,
    weekly["SaleAmt_Final_Sum"] / weekly["SaleAmt_Final_Count"],
    np.nan
)

weekly["BuyFee_RPU"] = np.where(
    weekly["Sold_Units"] > 0,
    weekly["BuyFee_Revenue"] / weekly["Sold_Units"],
    np.nan
)

weekly["BuyFee_Coverage"] = np.where(
    weekly["Sold_Units"] > 0,
    weekly["Sold_With_BuyFee"] / weekly["Sold_Units"],
    np.nan
)

weekly["Positive_BuyFee_Coverage"] = np.where(
    weekly["Sold_Units"] > 0,
    weekly["Sold_With_Positive_BuyFee"] / weekly["Sold_Units"],
    np.nan
)

weekly["Zero_BuyFee_Share"] = np.where(
    weekly["Sold_Units"] > 0,
    weekly["Sold_With_Zero_BuyFee"] / weekly["Sold_Units"],
    np.nan
)

for out_col, sum_col, count_col in [
    ("Avg_Vehicle_Age", "Vehicle_Age_Sum", "Vehicle_Age_Count"),
    ("Avg_Mileage", "Mileage_Sum", "Mileage_Count"),
    ("Avg_Condition", "Condition_Sum", "Condition_Count"),
    ("Avg_High_Bid", "High_Bid_Sum", "High_Bid_Count"),
    ("Avg_Floor", "Floor_Sum", "Floor_Count"),
]:
    weekly[out_col] = np.where(
        weekly[count_col] > 0,
        weekly[sum_col] / weekly[count_col],
        np.nan
    )

weekly["Year"] = weekly["Week_Start"].dt.year
weekly["Month"] = weekly["Week_Start"].dt.month
weekly["Quarter"] = weekly["Week_Start"].dt.quarter
weekly["Week_of_Year"] = (
    weekly["Week_Start"].dt.isocalendar().week.astype(int)
)

weekly = weekly.sort_values(
    ["VRLOCATION", "LANE", "price_tier_low", "Week_Start"]
).reset_index(drop=True)

weekly.to_csv(
    OUTPUT_DIR / "04_WEEKLY_MODEL_PANEL_V6_1.csv",
    index=False
)

print("Weekly rows:", f"{len(weekly):,}")
print("Weekly locations:", weekly["VRLOCATION"].nunique())
print("Total registered:", f"{weekly['Registered_Units'].sum():,}")
print("Total sold:", f"{weekly['Sold_Units'].sum():,}")
print(
    "Overall conversion:",
    round(
        weekly["Sold_Units"].sum()
        / weekly["Registered_Units"].sum(),
        4
    )
)
print("Median weekly registered:", weekly["Registered_Units"].median())
print("Median weekly sold:", weekly["Sold_Units"].median())


Weekly rows: 1,689,856
Weekly locations: 54
Total registered: 5,465,025
Total sold: 3,217,772
Overall conversion: 0.5888
Median weekly registered: 2.0
Median weekly sold: 1.0


## 8. Clean fee-change events + explicit hierarchy

In [9]:

events_raw = pd.concat(event_parts, ignore_index=True)

event_key = [
    "VRLOCATION",
    "LANE",
    "price_tier",
    "BUYFEE_CHANGE_DATE"
]

events = (
    events_raw.groupby(
        event_key,
        as_index=False,
        dropna=False
    )
    .agg(
        price_tier_low=("price_tier_low", "first"),
        price_tier_high=("price_tier_high", "first"),
        PREVIOUS_BUYFEE=("PREVIOUS_BUYFEE", "median"),
        BUYFEE_CHANGE_AMOUNT=("BUYFEE_CHANGE_AMOUNT", "median"),
        NEW_BUYFEE=("NEW_BUYFEE", "median"),
        Source_Rows=("NEW_BUYFEE", "size"),
    )
)

events["BUYFEE_CHANGE_DATE"] = pd.to_datetime(
    events["BUYFEE_CHANGE_DATE"],
    errors="coerce"
)

events["FEE_CHANGE_PCT"] = np.where(
    events["PREVIOUS_BUYFEE"].abs() > 0,
    events["BUYFEE_CHANGE_AMOUNT"] / events["PREVIOUS_BUYFEE"],
    np.nan
)

events["FEE_DIRECTION"] = np.select(
    [
        events["BUYFEE_CHANGE_AMOUNT"] > 0,
        events["BUYFEE_CHANGE_AMOUNT"] < 0,
    ],
    ["Increase", "Decrease"],
    default="No Change"
)

events["Corporate_Change_ID"] = (
    events["BUYFEE_CHANGE_DATE"].dt.strftime("%Y-%m-%d")
)

events["Location_Change_ID"] = (
    events["VRLOCATION"].astype(str)
    + "_"
    + events["Corporate_Change_ID"]
)

events["Segment_Change_ID"] = (
    events["VRLOCATION"].astype(str)
    + "_"
    + events["LANE"].astype(str)
    + "_"
    + events["price_tier"].astype(str)
    + "_"
    + events["Corporate_Change_ID"]
)

events = events.sort_values(
    ["VRLOCATION", "LANE", "price_tier", "BUYFEE_CHANGE_DATE"]
).reset_index(drop=True)

events["Prev_Change_Date"] = (
    events.groupby(
        ["VRLOCATION", "LANE", "price_tier"]
    )["BUYFEE_CHANGE_DATE"].shift(1)
)

events["Next_Change_Date"] = (
    events.groupby(
        ["VRLOCATION", "LANE", "price_tier"]
    )["BUYFEE_CHANGE_DATE"].shift(-1)
)

events["Weeks_Since_Prior_Change"] = (
    (
        events["BUYFEE_CHANGE_DATE"]
        - events["Prev_Change_Date"]
    ).dt.days / 7
)

events["Weeks_To_Next_Change"] = (
    (
        events["Next_Change_Date"]
        - events["BUYFEE_CHANGE_DATE"]
    ).dt.days / 7
)

events["Overlap_Flag"] = (
    events["Weeks_Since_Prior_Change"].lt(OVERLAP_LOOKBACK_WEEKS)
    |
    events["Weeks_To_Next_Change"].lt(OVERLAP_LOOKAHEAD_WEEKS)
).fillna(False)

events.to_csv(
    OUTPUT_DIR / "05_CLEAN_FEE_CHANGE_EVENTS_V6_1.csv",
    index=False
)

print("Segment fee events:", f"{len(events):,}")
print("Unique event locations:", events["VRLOCATION"].nunique())
print(
    "Location x Change Date:",
    events[["VRLOCATION", "BUYFEE_CHANGE_DATE"]]
    .drop_duplicates()
    .shape[0]
)
print(
    "Corporate change dates:",
    events["BUYFEE_CHANGE_DATE"].nunique()
)


Segment fee events: 9,251
Unique event locations: 38
Location x Change Date: 103
Corporate change dates: 51


## 9. Re-run 4W / 8W / 12W event pipeline

In [10]:

panel_lookup = {
    key: grp.sort_values("Week_Start").copy()
    for key, grp in weekly.groupby(
        ["VRLOCATION", "LANE", "price_tier"],
        sort=False
    )
}

def summarize_period(df):

    if df.empty:
        return {
            "observed_weeks": 0,
            "registered": 0,
            "sold": 0,
            "conversion": np.nan,
            "buyfee_revenue": 0.0,
            "buyfee_rpu": np.nan,
            "buyfee_coverage": np.nan,
            "positive_buyfee_coverage": np.nan,
            "zero_buyfee_share": np.nan,
            "avg_saleamt": np.nan,
            "avg_age": np.nan,
            "avg_mileage": np.nan,
            "avg_condition": np.nan,
            "avg_high_bid": np.nan,
            "avg_floor": np.nan,
            "volume_trend": np.nan,
            "conversion_trend": np.nan,
            "volume_cv": np.nan,
            "conversion_cv": np.nan,
        }

    reg = df["Registered_Units"].sum()
    sold = df["Sold_Units"].sum()
    fee_rev = df["BuyFee_Revenue"].sum()
    sold_with_fee = df["Sold_With_BuyFee"].sum()
    sold_positive_fee = df["Sold_With_Positive_BuyFee"].sum()
    sold_zero_fee = df["Sold_With_Zero_BuyFee"].sum()

    def wmean(sum_col, count_col):
        den = df[count_col].sum()
        return df[sum_col].sum() / den if den > 0 else np.nan

    return {
        "observed_weeks": df["Week_Start"].nunique(),
        "registered": reg,
        "sold": sold,
        "conversion": sold / reg if reg > 0 else np.nan,
        "buyfee_revenue": fee_rev,
        "buyfee_rpu": fee_rev / sold if sold > 0 else np.nan,
        "buyfee_coverage": sold_with_fee / sold if sold > 0 else np.nan,
        "positive_buyfee_coverage": (
            sold_positive_fee / sold if sold > 0 else np.nan
        ),
        "zero_buyfee_share": (
            sold_zero_fee / sold if sold > 0 else np.nan
        ),
        "avg_saleamt": wmean(
            "SaleAmt_Final_Sum",
            "SaleAmt_Final_Count"
        ),
        "avg_age": wmean(
            "Vehicle_Age_Sum",
            "Vehicle_Age_Count"
        ),
        "avg_mileage": wmean(
            "Mileage_Sum",
            "Mileage_Count"
        ),
        "avg_condition": wmean(
            "Condition_Sum",
            "Condition_Count"
        ),
        "avg_high_bid": wmean(
            "High_Bid_Sum",
            "High_Bid_Count"
        ),
        "avg_floor": wmean(
            "Floor_Sum",
            "Floor_Count"
        ),
        "volume_trend": linear_slope(
            df["Sold_Units"].values
        ),
        "conversion_trend": linear_slope(
            df["Conversion"].values
        ),
        "volume_cv": coefficient_of_variation(
            df["Sold_Units"].values
        ),
        "conversion_cv": coefficient_of_variation(
            df["Conversion"].values
        ),
    }

rows = []

for i, e in events.iterrows():

    key = (
        e["VRLOCATION"],
        e["LANE"],
        e["price_tier"]
    )

    seg = panel_lookup.get(key)

    if seg is None or seg.empty:
        continue

    event_date = pd.Timestamp(
        e["BUYFEE_CHANGE_DATE"]
    )

    event_week = (
        event_date
        - pd.to_timedelta(
            event_date.weekday(),
            unit="D"
        )
    )

    out = e.to_dict()
    out["Event_Week"] = event_week

    for w in WINDOWS:

        pre = seg[
            seg["Week_Start"].between(
                event_week - pd.Timedelta(weeks=w),
                event_week - pd.Timedelta(weeks=1)
            )
        ]

        post = seg[
            seg["Week_Start"].between(
                event_week + pd.Timedelta(weeks=1),
                event_week + pd.Timedelta(weeks=w)
            )
        ]

        a = summarize_period(pre)
        b = summarize_period(post)

        for k, v in a.items():
            out[f"pre_{w}w_{k}"] = v

        for k, v in b.items():
            out[f"post_{w}w_{k}"] = v

        out[f"volume_change_pct_{w}w"] = safe_pct_change(
            b["sold"], a["sold"]
        )

        out[f"registered_change_pct_{w}w"] = safe_pct_change(
            b["registered"], a["registered"]
        )

        out[f"conversion_change_pp_{w}w"] = (
            (b["conversion"] - a["conversion"]) * 100
            if pd.notna(a["conversion"])
            and pd.notna(b["conversion"])
            else np.nan
        )

        out[f"rpu_dollar_change_{w}w"] = (
            b["buyfee_rpu"] - a["buyfee_rpu"]
            if pd.notna(a["buyfee_rpu"])
            and pd.notna(b["buyfee_rpu"])
            else np.nan
        )

        out[f"fee_pass_through_{w}w"] = (
            out[f"rpu_dollar_change_{w}w"]
            / e["BUYFEE_CHANGE_AMOUNT"]
            if pd.notna(out[f"rpu_dollar_change_{w}w"])
            and pd.notna(e["BUYFEE_CHANGE_AMOUNT"])
            and e["BUYFEE_CHANGE_AMOUNT"] != 0
            else np.nan
        )

    rows.append(out)

    if (i + 1) % 1000 == 0:
        print(f"Processed {i+1:,} events")

event_model = pd.DataFrame(rows)

event_model.to_csv(
    OUTPUT_DIR / "06_EVENT_LEVEL_MODEL_DATA_RAW_V6_1.csv",
    index=False
)

print("Event-level rows:", f"{len(event_model):,}")


Processed 1,000 events
Processed 2,000 events
Processed 3,000 events
Processed 4,000 events
Processed 5,000 events
Processed 6,000 events
Processed 7,000 events
Processed 8,000 events
Processed 9,000 events
Event-level rows: 9,251


## 10. Reapply V6 eligibility rules

In [11]:

w = PRIMARY_WINDOW

event_model["Observed_Week_Eligible"] = (
    (event_model[f"pre_{w}w_observed_weeks"] >= MIN_OBSERVED_WEEKS_PRE)
    &
    (event_model[f"post_{w}w_observed_weeks"] >= MIN_OBSERVED_WEEKS_POST)
)

event_model["Registered_Exposure_Eligible"] = (
    (event_model[f"pre_{w}w_registered"] >= MIN_REGISTERED_PRE)
    &
    (event_model[f"post_{w}w_registered"] >= MIN_REGISTERED_POST)
)

event_model["Volume_Eligible"] = (
    event_model["Observed_Week_Eligible"]
    &
    event_model["Registered_Exposure_Eligible"]
    &
    (event_model[f"pre_{w}w_sold"] > 0)
    &
    event_model[f"volume_change_pct_{w}w"].notna()
)

event_model["Conversion_Eligible"] = (
    event_model["Observed_Week_Eligible"]
    &
    event_model["Registered_Exposure_Eligible"]
    &
    event_model[f"conversion_change_pp_{w}w"].notna()
)

event_model["RPU_Eligible"] = (
    event_model["Observed_Week_Eligible"]
    &
    event_model["Registered_Exposure_Eligible"]
    &
    (event_model[f"pre_{w}w_sold"] >= MIN_SOLD_PRE_RPU)
    &
    (event_model[f"post_{w}w_sold"] >= MIN_SOLD_POST_RPU)
    &
    (event_model[f"pre_{w}w_buyfee_coverage"] >= MIN_BUYFEE_COVERAGE)
    &
    (event_model[f"post_{w}w_buyfee_coverage"] >= MIN_BUYFEE_COVERAGE)
    &
    event_model[f"rpu_dollar_change_{w}w"].notna()
)

event_model["Primary_Model_Eligible"] = (
    event_model["Volume_Eligible"]
    &
    event_model["Conversion_Eligible"]
)

event_model["Strict_Model_Eligible"] = (
    event_model["Primary_Model_Eligible"]
    &
    ~event_model["Overlap_Flag"]
)

primary = event_model[
    event_model["Primary_Model_Eligible"]
].copy()

strict = event_model[
    event_model["Strict_Model_Eligible"]
].copy()

rpu_ready = event_model[
    event_model["RPU_Eligible"]
].copy()

event_model.to_csv(
    OUTPUT_DIR / "07_EVENT_LEVEL_WITH_FLAGS_V6_1.csv",
    index=False
)

primary.to_csv(
    OUTPUT_DIR / "08_MODEL_READY_PRIMARY_V6_1.csv",
    index=False
)

strict.to_csv(
    OUTPUT_DIR / "09_MODEL_READY_STRICT_V6_1.csv",
    index=False
)

rpu_ready.to_csv(
    OUTPUT_DIR / "10_MODEL_READY_RPU_V6_1.csv",
    index=False
)

print("Observed week eligible:", f"{event_model['Observed_Week_Eligible'].sum():,}")
print("Registered eligible:", f"{event_model['Registered_Exposure_Eligible'].sum():,}")
print("Volume eligible:", f"{event_model['Volume_Eligible'].sum():,}")
print("Conversion eligible:", f"{event_model['Conversion_Eligible'].sum():,}")
print("RPU eligible:", f"{event_model['RPU_Eligible'].sum():,}")
print("Primary eligible:", f"{len(primary):,}")
print("Strict eligible:", f"{len(strict):,}")


Observed week eligible: 4,062
Registered eligible: 2,221
Volume eligible: 2,142
Conversion eligible: 2,142
RPU eligible: 504
Primary eligible: 2,142
Strict eligible: 1,843


## 11. Scenario support after corrected sold logic

In [ ]:

scenario_rows = []

for target in [10, 15, 20, 25, 30]:

    all_support = events[
        events["BUYFEE_CHANGE_AMOUNT"].between(
            target - 2.5,
            target + 2.5
        )
    ]

    primary_support = primary[
        primary["BUYFEE_CHANGE_AMOUNT"].between(
            target - 2.5,
            target + 2.5
        )
    ]

    strict_support = strict[
        strict["BUYFEE_CHANGE_AMOUNT"].between(
            target - 2.5,
            target + 2.5
        )
    ]

    scenario_rows.append({
        "Scenario_Fee_Increase": target,

        "All_Segment_Events": len(all_support),
        "All_Unique_Locations": all_support["VRLOCATION"].nunique(),
        "All_Corporate_Change_Dates": all_support["BUYFEE_CHANGE_DATE"].nunique(),

        "Primary_Eligible_Events": len(primary_support),
        "Primary_Unique_Locations": primary_support["VRLOCATION"].nunique(),
        "Primary_Corporate_Change_Dates": primary_support["BUYFEE_CHANGE_DATE"].nunique(),

        "Strict_Eligible_Events": len(strict_support),
        "Strict_Unique_Locations": strict_support["VRLOCATION"].nunique(),
        "Strict_Corporate_Change_Dates": strict_support["BUYFEE_CHANGE_DATE"].nunique(),
    })

scenario_support = pd.DataFrame(scenario_rows)

scenario_support.to_csv(
    OUTPUT_DIR / "11_SCENARIO_SUPPORT_V6_1.csv",
    index=False
)

display(scenario_support)


,Scenario_Fee_Increase,All_Segment_Events,All_Unique_Locations,All_Corporate_Change_Dates,Primary_Eligible_Events,Primary_Unique_Locations,Primary_Corporate_Change_Dates,Strict_Eligible_Events,Strict_Unique_Locations,Strict_Corporate_Change_Dates
0,10,2600,18,31,774,16,25,597,16,24
1,15,329,17,16,89,14,10,85,14,10
2,20,1034,17,19,141,14,10,134,13,10
3,25,467,18,13,43,10,4,43,10,4
4,30,323,17,17,65,8,4,60,8,4


## 12. Final validation summary and comparison-ready diagnostics

In [13]:

sold_total = int(sold_recon["Sold_Flag_Count"].sum())

sold_fee_populated = int(
    sold_fee["Populated_Count"].sum()
)

sold_fee_positive = int(
    sold_fee["POSITIVE"].sum()
)

sold_fee_zero = int(
    sold_fee["ZERO"].sum()
)

sold_fee_null = int(
    sold_fee["NULL"].sum()
)

summary = pd.DataFrame([
    {"Metric": "Raw_Rows_Processed", "Value": rows_processed},

    {"Metric": "All_Location_Labels", "Value": location_audit["VRLOCATION"].nunique()},
    {"Metric": "Valid_Inventory_Locations", "Value": valid_locations["VRLOCATION"].nunique()},
    {"Metric": "Expected_Business_Locations", "Value": EXPECTED_LOCATION_COUNT},
    {
        "Metric": "Location_Count_Difference",
        "Value": valid_locations["VRLOCATION"].nunique() - EXPECTED_LOCATION_COUNT
    },

    {"Metric": "Sold_Rows_NULL_Safe_Logic", "Value": sold_total},

    {"Metric": "Sold_BUYFEE_FINAL_Populated", "Value": sold_fee_populated},
    {"Metric": "Sold_BUYFEE_FINAL_Positive", "Value": sold_fee_positive},
    {"Metric": "Sold_BUYFEE_FINAL_Zero", "Value": sold_fee_zero},
    {"Metric": "Sold_BUYFEE_FINAL_NULL", "Value": sold_fee_null},
    {
        "Metric": "Sold_BUYFEE_FINAL_Coverage",
        "Value": sold_fee_populated / sold_total if sold_total else np.nan
    },

    {"Metric": "Weekly_Panel_Rows", "Value": len(weekly)},
    {"Metric": "Weekly_Panel_Locations", "Value": weekly["VRLOCATION"].nunique()},
    {"Metric": "Weekly_Total_Registered", "Value": weekly["Registered_Units"].sum()},
    {"Metric": "Weekly_Total_Sold", "Value": weekly["Sold_Units"].sum()},
    {
        "Metric": "Portfolio_Conversion",
        "Value": (
            weekly["Sold_Units"].sum()
            / weekly["Registered_Units"].sum()
            if weekly["Registered_Units"].sum() else np.nan
        )
    },

    {"Metric": "Segment_Fee_Events", "Value": len(events)},
    {"Metric": "Fee_Event_Locations", "Value": events["VRLOCATION"].nunique()},
    {
        "Metric": "Location_x_Change_Date_Events",
        "Value": events[
            ["VRLOCATION", "BUYFEE_CHANGE_DATE"]
        ].drop_duplicates().shape[0]
    },
    {
        "Metric": "Corporate_Change_Dates",
        "Value": events["BUYFEE_CHANGE_DATE"].nunique()
    },

    {
        "Metric": "Observed_Week_Eligible",
        "Value": int(event_model["Observed_Week_Eligible"].sum())
    },
    {
        "Metric": "Registered_Exposure_Eligible",
        "Value": int(event_model["Registered_Exposure_Eligible"].sum())
    },
    {
        "Metric": "Volume_Eligible",
        "Value": int(event_model["Volume_Eligible"].sum())
    },
    {
        "Metric": "Conversion_Eligible",
        "Value": int(event_model["Conversion_Eligible"].sum())
    },
    {
        "Metric": "RPU_Eligible",
        "Value": int(event_model["RPU_Eligible"].sum())
    },

    {"Metric": "Primary_Model_Eligible", "Value": len(primary)},
    {"Metric": "Primary_Model_Unique_Locations", "Value": primary["VRLOCATION"].nunique()},
    {
        "Metric": "Primary_Model_Corporate_Change_Dates",
        "Value": primary["BUYFEE_CHANGE_DATE"].nunique()
    },

    {"Metric": "Strict_Model_Eligible", "Value": len(strict)},
    {"Metric": "Strict_Model_Unique_Locations", "Value": strict["VRLOCATION"].nunique()},
    {
        "Metric": "Strict_Model_Corporate_Change_Dates",
        "Value": strict["BUYFEE_CHANGE_DATE"].nunique()
    },

    {"Metric": "RPU_Model_Eligible", "Value": len(rpu_ready)},
    {"Metric": "RPU_Model_Unique_Locations", "Value": rpu_ready["VRLOCATION"].nunique()},
])

summary.to_csv(
    OUTPUT_DIR / "12_VALIDATION_SUMMARY_V6_1.csv",
    index=False
)

display(summary)


,Metric,Value
0,Raw_Rows_Processed,5.540598e+06
1,All_Location_Labels,5.400000e+01
2,Valid_Inventory_Locations,5.400000e+01
3,Expected_Business_Locations,5.200000e+01
4,Location_Count_Difference,2.000000e+00
5,Sold_Rows_NULL_Safe_Logic,3.217772e+06
6,Sold_BUYFEE_FINAL_Populated,2.024248e+06
7,Sold_BUYFEE_FINAL_Positive,2.016137e+06
8,Sold_BUYFEE_FINAL_Zero,8.111000e+03
9,Sold_BUYFEE_FINAL_NULL,1.193524e+06


## 13. QA checks

In [14]:

sold_recon_ok = bool(
    (sold_recon["Sold_Reconciliation_Difference"] == 0).all()
)

valid_loc_count = valid_locations["VRLOCATION"].nunique()

qa = pd.DataFrame([
    {
        "Check": "NULL-safe HST/SLD sold logic",
        "Status": "PASS" if sold_recon_ok else "REVIEW",
        "Detail": (
            "Blank invoice/IFORTRY values are treated as empty strings "
            "before HST/SLD business logic."
        )
    },
    {
        "Check": "Sold reconciliation",
        "Status": "PASS" if sold_recon_ok else "REVIEW",
        "Detail": "Sold_Flag_Count must equal business-rule expected sold count by status."
    },
    {
        "Check": "Location reconciliation",
        "Status": "PASS" if valid_loc_count == EXPECTED_LOCATION_COUNT else "REVIEW",
        "Detail": (
            f"Source has {valid_loc_count} valid-inventory location labels; "
            f"business expectation is {EXPECTED_LOCATION_COUNT}. "
            "Use 03_LOCATION_RECONCILIATION_V6_1.csv to identify aliases/legacy locations."
        )
    },
    {
        "Check": "BUYFEE_FINAL source",
        "Status": "PASS",
        "Detail": "Only BUYFEE_FINAL is used for Buy Fee revenue/RPU."
    },
    {
        "Check": "BUYFEE_FINAL coverage",
        "Status": "REVIEW",
        "Detail": (
            "Coverage is explicitly split into NULL / ZERO / POSITIVE. "
            "Review 02_BUYFEE_FINAL_COVERAGE_V6_1.csv before final RPU modeling."
        )
    },
    {
        "Check": "SALEAMT_FINAL tiering",
        "Status": "PASS",
        "Detail": "SALEAMT_FINAL is used for sold and unsold price tiers."
    },
    {
        "Check": "INV/SAL exclusion",
        "Status": "PASS",
        "Detail": "Only HST, SLD and NOS enter the standard model inventory."
    },
    {
        "Check": "6-of-8 observed weeks",
        "Status": "PASS",
        "Detail": "Primary 8W event eligibility requires at least six observed weeks pre and post."
    },
    {
        "Check": ">=25 registered units",
        "Status": "PASS",
        "Detail": "Primary 8W event eligibility requires at least 25 registered units pre and post."
    },
])

qa.to_csv(
    OUTPUT_DIR / "13_QA_CHECKLIST_V6_1.csv",
    index=False
)

display(qa)


,Check,Status,Detail
0,NULL-safe HST/SLD sold logic,PASS,Blank invoice/IFORTRY values are treated as em...
1,Sold reconciliation,PASS,Sold_Flag_Count must equal business-rule expec...
2,Location reconciliation,REVIEW,Source has 54 valid-inventory location labels;...
3,BUYFEE_FINAL source,PASS,Only BUYFEE_FINAL is used for Buy Fee revenue/...
4,BUYFEE_FINAL coverage,REVIEW,Coverage is explicitly split into NULL / ZERO ...
5,SALEAMT_FINAL tiering,PASS,SALEAMT_FINAL is used for sold and unsold pric...
6,INV/SAL exclusion,PASS,"Only HST, SLD and NOS enter the standard model..."
7,6-of-8 observed weeks,PASS,Primary 8W event eligibility requires at least...
8,>=25 registered units,PASS,Primary 8W event eligibility requires at least...



## Files to share back

Please upload these after V6.1 finishes:

1. `01_SOLD_LOGIC_RECONCILIATION_V6_1.csv`
2. `02_BUYFEE_FINAL_COVERAGE_V6_1.csv`
3. `03_LOCATION_RECONCILIATION_V6_1.csv`
4. `11_SCENARIO_SUPPORT_V6_1.csv`
5. `12_VALIDATION_SUMMARY_V6_1.csv`
6. `13_QA_CHECKLIST_V6_1.csv`
7. `08_MODEL_READY_PRIMARY_V6_1.csv`
8. `10_MODEL_READY_RPU_V6_1.csv`

The first six are the most important.

### What we should expect if the NULL fix solved the main V6 issue

- Sold rows should move much closer to the HST population minus actual RETURN records, plus qualifying SLD.
- Portfolio conversion should become operationally plausible.
- Many more than two locations should survive the primary event rules.
- The +$10 / +$15 / +$20 / +$25 / +$30 eligible support should materially improve.
- RPU eligibility will then depend primarily on actual BUYFEE_FINAL coverage rather than the sold-flag bug.
